In [1]:
%cd final_pipe 

/home/jovyan/Group15_innovation/entity-linking/final_pipe


# Patent Applicant Name Harmonization — End-to-End Production Pipeline

This notebook implements a **three-stage entity-resolution pipeline** that maps raw PATSTAT patent applicant strings (`psn_name`) into canonical corporate, academic, and public research entities (`harm_id_final`, `harmonized_name_final`) without requiring labeled training pairs.

### Pipeline Architecture
1. **Stage 0 — EPO/PATSTAT Sector Imputation & Filtering:** Replaces `UNKNOWN` or missing `psn_sector` labels using multilingual corporate, academic, government/non-profit, and personal-name rules so legitimate organizations labeled `UNKNOWN` are retained while individual inventors are excluded.
2. **Stage 1 — Generic Multi-Signal Graph Baseline:** Normalizes Unicode/CJK/legal suffixes, computes a balanced similarity score combining **Multilingual SBERT Cosine** (`paraphrase-multilingual-MiniLM-L12-v2`), **IDF-Weighted Jaccard**, and **Character Edit Ratio**, finds an unsupervised cutoff via **Otsu thresholding**, and partitions the similarity graph via **Louvain community detection**.
3. **Stage 2 — Sector-Stratified Hairball Rectification:** Audits every multi-member Stage 1 cluster (`size >= 2`), enforces consistent `entity_type` boundaries (`company`, `university`, `gov / non-profit`), applies squared-IDF core token guards (`CORE_JAC_MIN = 0.35`, `CORE_EDIT_MIN = 0.80`), and re-partitions oversized/mixed clusters (`resolution = 1.35`) to eliminate transitive chain-merges (`HONDA` + `TOSHIBA`, `WILLIS` + `INSPUR`, `SIEMENS` + `TENCENT`).
4. **Stage 3 — Core-Anchor & Subsidiary Consolidation:** Recovers recall across regional subsidiaries, word-order swaps, and exact acronym expansions (`IBM` $\leftrightarrow$ `IBM DEUTSCHLAND` / `INTERNATIONAL BUSINESS MACHINES`, `STANFORD UNIV` $\leftrightarrow$ `LELAND STANFORD JUNIOR UNIV`) within compatible entity categories.

## 1. Imports & Pipeline Configuration

Centralizes all input/output paths, embedding model settings, Stage 1 similarity weights, Stage 2 rectification thresholds, and Stage 3 anchor parameters for full reproducibility.

In [2]:
%pip install rapidfuzz sentence_transformers


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import os
import unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
import networkx.algorithms.community as nx_comm
from rapidfuzz.fuzz import ratio as rf_ratio, token_sort_ratio as rf_token_sort
from sentence_transformers import SentenceTransformer

# --- Input / Output Files ---
BASE_DIR = "/home/jovyan/Group15_innovation/entity-linking"
INPUT_CSV = os.path.join(BASE_DIR, "qc_applicants_PROD.csv")

OUTPUT_FOLDER =  "outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
OUTPUT_STAGE1_LOOKUP = os.path.join(OUTPUT_FOLDER, "harmonization_lookup_stage1.csv")
OUTPUT_FINAL_LOOKUP = os.path.join(OUTPUT_FOLDER, "harmonization_lookup_rectified.csv")
OUTPUT_FINAL_DATA = os.path.join(OUTPUT_FOLDER, "harmonization_output.csv")

NAME_COL = "psn_name"
SECTOR_COL = "psn_sector"
WEIGHT_COL = None  # optional: e.g. "n_families" if available

# --- Stage 1: Generic Multi-Signal Graph Parameters ---
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
SCORE_WEIGHTS = {"cosine": 0.34, "jaccard": 0.33, "edit": 0.33}
SCORE_FLOOR = 0.45
MIN_OTSU_FLOOR = 0.56
LOUVAIN_RESOLUTION_S1 = 1.5
LOUVAIN_SEED = 42

# --- Stage 2: Hairball Rectification Parameters ---
MIN_CLUSTER_SIZE_TO_SPLIT = 2
CORE_JAC_MIN = 0.35
CORE_EDIT_MIN = 0.80
LOUVAIN_RESOLUTION_S2 = 1.35

# --- Stage 3: Core-Anchor Consolidation Parameters ---
RARE_ANCHOR_MAX_DF = 12.0
CORE_FUZZY_MIN = 90.0

## 2. EPO/PATSTAT Sector Imputation & Deterministic Text Normalization

Before building similarity graphs, two preprocessing operations run on the raw applicant table:
1. **EPO/PATSTAT Sector Imputation (`impute_psn_sector`):** Preserves known PATSTAT `psn_sector` tags (`COMPANY`, `UNIVERSITY`, `GOV NON-PROFIT`, `INDIVIDUAL`) and classifies `UNKNOWN`/missing rows via multilingual legal forms (`GMBH`, `SPA`, `LTD`, `株式会社`, `有限公司`), academic markers (`UNIV`, `ECOLE`, `HOCHSCHULE`, `大学`), public/research institute markers (`CNRS`, `FRAUNHOFER`, `MINISTRY`, `学院`), corporate technology markers, and personal-name patterns. Unlabeled individuals are dropped while all organizational `UNKNOWN` rows are retained.
2. **Deterministic Name Cleaning (`clean`):** Folds Unicode/diacritics, extracts Latin equivalents appended to CJK names, strips `C/O` care-of clauses and incorporation boilerplate, standardizes institutional terms (`UNIVERSITY` $\rightarrow$ `UNIV`, `TECHNOLOGY` $\rightarrow$ `TECH`), drops filler prepositions/articles, and removes leading/trailing legal entity suffixes.

In [4]:
# --- Multilingual Organizational & Personal Name Patterns (EPO / PATSTAT Sector Imputation) ---
LEGAL_SEC = [
    "INC", "LLC", "LLLP", "LL", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "COPMANY",
    "GMBH", "AG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "NA", "OY", "AB", "AS",
    "ASA", "PTY", "PLC", "KK", "LLP", "LP", "PTE", "KG", "OOO", "SP Z OO", "SPOLKA", "JOINT.STOCK",
    "ANONIM SIRKETI", "OBSHCHESTVO"
]
LEGAL_CJK_SEC = ["有限公司", "株式会社", "股份", "公司", "会社", "주식회사", "インコーポレイテッド", "アクチボラグ"]

UNIV_SEC = [
    "UNIV", "COLLEGE", "POLYTECHN", "POLITECNIC", "ECOLE", "HOCHSCHULE", "SCHOOL",
    "INSTITUTE OF TECHNOLOGY", "GIJUKU", "TRUSTEES", "EDUCATIONAL FUND",
    "大学", "大學", "学院", "カレッジ", "義塾", r"\bETH\b", r"\bEPFL\b", r"\bUESTC\b"
]

GOV_SEC = [
    "INSTIT", "INSTYTUT", "INSTUTUC", "ИНСТИТУТ", "ACADEM", "COMMISSARIAT", "CNRS", "LABORATORY",
    "CENTER", "CENTRE", "CENTRUM", "ZENTRUM", "FOUNDATION", "FONDAZIONE", "FUNDACIO", "STIFTUNG",
    "FRAUNHOFER", "FROUNHAV", "PLANCK", "GOVERNMENT", "MINISTRY", "MINISTERIUM", "BUNDES",
    "FEDERATSIYA", "AGENTSTVO", "UCHREZHDENIE", "SECRETARY", "AGENCY", "COUNCIL", "CONSEJO",
    "CONSIGLIO", r"\bNATIONAL\b", "IDRYMA", "ASSOCIATION", "HOSPITAL", "ARMY", "AIR FORCE", "NAVY",
    "RIKEN", r"\bTNO\b", "NASA", "研究", "中心", "委员会", "科学院", "センター"
]

FIRM_SEC = [
    "TECHNOLOG", "QUANT", "COMPUT", "SYSTEM", "SOLUTION", r"\bLABS?\b", "SOFTWARE", "NETWORK",
    "ELECTRONIC", "SEMICONDUCT", "PHOTONIC", "DEVICES", "INDUSTR", "HOLDING", "GROUP", "CAPITAL",
    "VENTURES", "ENTERPRISE", "INTERNATIONAL", "SCIENCE", "DIGITAL", "DATA", "INTELLIGEN", "ENERGY",
    "PHARMA", "SECUR", "MACHINES", "INSTRUMENTS", "ENGINEERING", "INNOVATION", "BLOCKCHAIN",
    "BANK", "INSURANCE", "PORTFOLIO", r"\bAI\b", "&", r"\+", "科技", "技术", "量子"
]

LEGAL_RE = re.compile(r"\b(" + "|".join(LEGAL_SEC) + r")\b|" + "|".join(LEGAL_CJK_SEC))
UNIV_RE, GOV_RE, FIRM_RE = (re.compile("|".join(words)) for words in (UNIV_SEC, GOV_SEC, FIRM_SEC))

WORD_PAT = r"[^\W\d_]+(?:['\-][^\W\d_]+)*"
PERSON_COMMA = re.compile(rf"^{WORD_PAT}( {WORD_PAT}){{0,3}} ?, ?{WORD_PAT}")
PERSON_WORDS = re.compile(rf"^{WORD_PAT}( {WORD_PAT}){{1,3}}$")


def _norm_for_sector(name: str) -> str:
    s = unicodedata.normalize("NFKC", str(name))
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)).upper()
    s = " ".join(s.replace("?", "").replace(".", "").split())
    return s.replace("UNIVERSITY CORPORATION", "UNIVERSITY")


def impute_psn_sector(psn_name: str, psn_sector: str) -> str:
    """Keep known PATSTAT sectors; replace UNKNOWN/NaN using EPO-style multilingual rules."""
    s = str(psn_sector).strip().upper() if pd.notna(psn_sector) else ""
    if s and s != "UNKNOWN":
        if "UNIVERSITY" in s:             return "UNIVERSITY"
        if "GOV" in s or "HOSPITAL" in s: return "GOV NON-PROFIT"
        if "COMPANY" in s:                return "COMPANY"
        if "INDIVIDUAL" in s:             return "INDIVIDUAL"
        return s

    n = _norm_for_sector(psn_name)
    if LEGAL_RE.search(n): return "COMPANY"
    if UNIV_RE.search(n):  return "UNIVERSITY"
    if GOV_RE.search(n):   return "GOV NON-PROFIT"
    if FIRM_RE.search(n):  return "COMPANY"
    if PERSON_COMMA.match(n) or PERSON_WORDS.match(n) or "·" in n:
        return "INDIVIDUAL"
    if len(n.split()) == 1:
        return "COMPANY"
    return "UNCERTAIN"


# --- Deterministic String Normalization ---
FORMS = [
    "INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION",
    "CO", "COMPANY", "PLC", "GMBH", "MBH", "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL",
    "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE", "KK",
    "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA", "LIMITEE", "LTEE",
    "AKTIENGESELLSCHAFT", "SOCIETE ANONYME", "SOCIEDAD ANONIMA", "SOCIETA PER AZIONI"
]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"
WORDS = {
    "UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV",
    "UNIVERSIDAD": "UNIV", "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST",
    "INSTITUTO": "INST", "TECHNOLOGY": "TECH", "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH",
    "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB", "CENTRE": "CENTER",
    "SCIENCES": "SCIENCE", "大學": "大学"
}
FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}


def clean(name):
    if pd.isna(name):
        return ""
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "ɐ")))
    s = re.sub(r"[▲▼?.'’]", "", s)
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
    s = re.sub(r"^C/O\s+", "", s).strip()
    s = re.sub(r"(?:,|\s)\s*\bC/O\b.*$", "", s)
    s = re.sub(r"\(\s*A\s+[A-Z ]+\b(?:CORP|CORPORATION|COMPANY|LIMITED|LTD|INC)\b[^)]*\)", " ", s)
    s = s.replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip())
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    s = re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s
    return s


# --- Load & Preprocess Raw Data ---
raw_df = pd.read_csv(INPUT_CSV)
raw_df["psn_sector_imputed"] = [
    impute_psn_sector(n, sec) for n, sec in zip(raw_df[NAME_COL], raw_df[SECTOR_COL])
]

# Known borderline overrides
raw_df.loc[raw_df[NAME_COL].isin([
    "DHITECH - DISTRETTO TECNOLOGICO H IGH TECH S.C.A R.L.",
    "PEARL COHEN ZEDEK LATZER BARATZ"
]), "psn_sector_imputed"] = "COMPANY"
raw_df.loc[raw_df[NAME_COL].str.contains("EHNSEHRM", na=False), "psn_sector_imputed"] = "GOV NON-PROFIT"

# Retain all organizational rows (including imputed UNKNOWN organizations); exclude individuals
df = raw_df[
    (raw_df["psn_sector_imputed"] != "INDIVIDUAL") &
    (raw_df[NAME_COL].notna()) &
    (raw_df[NAME_COL].str.strip() != "-NOT AVAILABLE-")
].copy()

df["clean"] = df[NAME_COL].apply(clean)
df = df[df["clean"].str.len() > 0].copy()
unique_names = sorted(df["clean"].unique())

print(f"Total raw rows: {len(raw_df):,} | Retained org rows: {len(df):,} | Unique cleaned names: {len(unique_names):,}")
print("\nImputed sector distribution across retained rows:")
print(df["psn_sector_imputed"].value_counts())

Total raw rows: 20,895 | Retained org rows: 17,751 | Unique cleaned names: 2,650

Imputed sector distribution across retained rows:
psn_sector_imputed
COMPANY           13159
UNIVERSITY         3473
GOV NON-PROFIT     1118
UNCERTAIN             1
Name: count, dtype: int64


## 3. Stage 1: Multi-Signal Semantic & Lexical Graph Clustering

Encodes all unique normalized organizational names with `paraphrase-multilingual-MiniLM-L12-v2`, computes pairwise composite similarity (`0.34 * cosine + 0.33 * idf_jaccard + 0.33 * edit_ratio`) with a zero-token-overlap typo guard (`ed >= 0.80`), automatically selects the bimodal cutoff via **Otsu thresholding**, and runs **Louvain community detection** (`resolution = 1.5`) to produce initial `HARM_XXXXX` clusters.

In [5]:
def pair_features_sector(names, emb, w=SCORE_WEIGHTS, floor=SCORE_FLOOR):
    n = len(names)
    wc, wj, we = w["cosine"], w["jaccard"], w["edit"]
    tokens = [set(s.split()) for s in names]
    lens = np.array([len(s) for s in names], dtype=np.float32)

    df_counts = Counter(t for toks in tokens for t in toks)
    idf = {t: np.log(n / c) for t, c in df_counts.items()}
    tok_idf_sums = np.array([sum(idf[t] for t in toks) for toks in tokens], dtype=np.float32)

    sim_matrix = emb @ emb.T
    min_cos = (floor - wj - we) / wc
    rows, cols = np.where(np.triu(sim_matrix >= min_cos, k=1))

    out = []
    for i, j in zip(rows, cols):
        cos = float(sim_matrix[i, j])
        ta, tb = tokens[i], tokens[j]
        inter = ta & tb
        if inter:
            inter_idf = sum(idf[t] for t in inter)
            union_idf = tok_idf_sums[i] + tok_idf_sums[j] - inter_idf
            jac = float(inter_idf / union_idf) if union_idf > 0 else 0.0
        else:
            jac = 0.0

        if wc * cos + wj * jac + we < floor:
            continue

        li, lj = lens[i], lens[j]
        max_ed = (2.0 * min(li, lj) / (li + lj)) if (li + lj) > 0 else 0.0
        if wc * cos + wj * jac + we * max_ed < floor:
            continue

        ed = rf_ratio(names[i], names[j]) / 100.0
        req_ed = max(0.0, (floor - wc * cos - wj * jac) / we)
        if jac == 0.0:
            req_ed = max(req_ed, 0.80)
        if req_ed > 1.0:
            continue

        score = wc * cos + wj * jac + we * ed
        if score >= floor:
            out.append((names[i], names[j], score))

    return pd.DataFrame(out, columns=["name_a", "name_b", "score"])


def find_threshold_otsu(scores, n_bins=100):
    counts, bin_edges = np.histogram(scores, bins=n_bins)
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    w1 = np.cumsum(counts)
    w2 = w1[-1] - w1
    valid = (w1 > 0) & (w2 > 0)
    w1, w2, centers_v = w1[valid], w2[valid], centers[valid]
    csum = np.cumsum(counts * centers)[valid]
    total_sum = np.sum(counts * centers)
    m1 = csum / w1
    m2 = (total_sum - csum) / w2
    between_var = w1 * w2 * (m1 - m2) ** 2
    return float(centers_v[np.argmax(between_var)])


model = SentenceTransformer(MODEL_NAME)
emb = model.encode(unique_names, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
emb_map = dict(zip(unique_names, emb))

scored = pair_features_sector(unique_names, emb)
raw_otsu = find_threshold_otsu(scored["score"].values) if len(scored) > 0 else MIN_OTSU_FLOOR
th_otsu = max(raw_otsu, MIN_OTSU_FLOOR)
print(f"Candidate edges: {len(scored):,} | Stage 1 Otsu threshold: {th_otsu:.3f}")

G = nx.Graph()
G.add_nodes_from(unique_names)
edges = scored[scored["score"] >= th_otsu]
G.add_weighted_edges_from(zip(edges["name_a"], edges["name_b"], edges["score"]))
communities = nx_comm.louvain_communities(G, weight="weight", resolution=LOUVAIN_RESOLUTION_S1, seed=LOUVAIN_SEED)

if WEIGHT_COL and WEIGHT_COL in df.columns:
    name_weights = df.groupby("clean")[WEIGHT_COL].sum().to_dict()
else:
    name_weights = df["clean"].value_counts().to_dict()

sorted_comms = sorted(communities, key=lambda c: (-sum(name_weights.get(m, 0) for m in c), min(c)))

clean_to_canon_s1 = {}
clean_to_id_s1 = {}
for idx, members in enumerate(sorted_comms, start=1):
    harm_id = f"HARM_{idx:05d}"
    best_name = min(members, key=lambda n: (-name_weights.get(n, 0), len(n), n))
    for m in members:
        clean_to_canon_s1[m] = best_name
        clean_to_id_s1[m] = harm_id

df["harm_id"] = df["clean"].map(clean_to_id_s1)
df["harmonized_name"] = df["clean"].map(clean_to_canon_s1)

lookup_s1 = (
    df[[NAME_COL, "clean", "psn_sector_imputed", "harm_id", "harmonized_name"]]
    .drop_duplicates()
    .sort_values(["harm_id", NAME_COL])
)
lookup_s1.to_csv(OUTPUT_STAGE1_LOOKUP, index=False)
print(f"Stage 1 complete: {len(unique_names):,} unique names -> {len(sorted_comms):,} initial clusters.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Candidate edges: 29,233 | Stage 1 Otsu threshold: 0.560
Stage 1 complete: 2,650 unique names -> 1,372 initial clusters.


## 4. Stage 2: Sector-Stratified Hairball Rectification

While Stage 1 accurately groups clear variants, multi-member Stage 1 clusters (`size >= 2`) can still suffer from transitive chain-merges driven by shared generic templates (e.g., `HONDA RESEARCH INST EUROPE` + `TOSHIBA RESEARCH EUROPE`, `WILLIS GROUP` + `INSPUR GROUP`, `UNIV DEGLI STUDI MILANO` + `UNIV DEGLI STUDI BOLOGNA`).

This stage inspects **every multi-member Stage 1 cluster** (`MIN_CLUSTER_SIZE_TO_SPLIT = 2`) and enforces:
1. **Consistent Entity-Type Partitioning:** Maps `psn_sector_imputed` to canonical `entity_type` (`company`, `university`, `gov / non-profit`, `uncertain`) by majority vote per unique `clean` string so identical strings never split across sectors, while blocking cross-sector merges inside Stage 1 clusters.
2. **Core-Token Squared-IDF Jaccard & Edit Guards:** Strips institutional, sector, and geographic boilerplate (`BOILERPLATE_CORE`), weights distinctive brand/institution tokens by $\text{idf}(t)^2$, and requires either `CORE_JAC >= 0.35` or character similarity `>= 0.80` before retaining an edge.
3. **High-Resolution Louvain Re-Partitioning (`resolution = 1.35`):** Splits mixed or chained clusters cleanly into pure sub-entities.

In [6]:
SECTOR_TO_ENTITY = {
    "COMPANY": "company",
    "UNIVERSITY": "university",
    "GOV NON-PROFIT": "gov / non-profit",
    "UNCERTAIN": "uncertain",
    "INDIVIDUAL": "individual",
}

df_work = lookup_s1.copy()
df_work["entity_type"] = (
    df_work["psn_sector_imputed"].str.strip().str.upper().map(SECTOR_TO_ENTITY).fillna("uncertain")
)

# Enforce consistent entity_type per unique cleaned string (prevents RIKEN / OXFORD UNIV INNOVATION splits)
clean_maj_type = (
    df_work.groupby(["clean", "entity_type"]).size()
    .reset_index(name="cnt")
    .sort_values(["clean", "cnt"], ascending=[True, False])
    .drop_duplicates("clean")
    .set_index("clean")["entity_type"]
)
df_work["entity_type"] = df_work["clean"].map(clean_maj_type)

# Corpus-wide IDF statistics on unique cleaned strings
N_CORPUS = len(unique_names)
df_counts = Counter(t for s in unique_names for t in set(s.split()))
idf_map = {t: np.log(N_CORPUS / c) for t, c in df_counts.items()}

BOILERPLATE_CORE = {
    # Institutional & Legal
    "UNIV", "INST", "INSTITUTE", "COLLEGE", "ACADEMY", "SCHOOL", "HOCHSCHULE", "ECOLE",
    "POLITECNICA", "POLYTECHNICAL", "POLYTECHNIC", "UNIVERSITETT", "UNIVERSITAET", "UNIVERSITAT",
    "UNIVERSITE", "UNIVERSIDAD", "UNIVERSITA", "UNIVERSITET", "TECHNISCHE", "VOCATIONAL",
    "TECH", "SCIENCE", "RESEARCH", "LAB", "CENTER", "GROUP", "CORP", "COMPANY", "COLTD",
    "HOLDINGS", "INTERNATIONAL", "NATIONAL", "STATE", "PUBLIC", "ROYAL", "NORMAL", "GENERAL",
    "CENTRAL", "FEDERAL", "LIMITED", "RESPONSIBILITY", "KORPERSCHAFT", "OFFENTLICHEN", "RECHTS",
    "BOARD", "REGENTS", "TRUSTEES", "FOUNDATION", "ASSOCIATION", "PRESIDENT", "FELLOWS",
    "HOSPITAL", "DEPARTMENT", "MINISTRY", "GOVERNMENT", "AGENCY", "SECRETARY", "DIRECTOR",
    "COMMITTEE", "COUNCIL", "INSTITUTION", "ADMINISTRATIVE", "INDEPENDENT",
    "DEGLI", "STUDI", "NORMALE", "SUPERIEURE", "POSTS", "COOPERATION", "ACADEMIC", "ACADEMIA",
    "FEDERALNOE", "GOSUDARSTVENNOE", "BYUDZHETNOE", "OBRAZOVATELNOE", "UCHREZHDENIE", "VYSSHEGO", "OBRAZOVANIYA",
    "GESELLSCHAFT", "FORDERUNG", "ANGEWANDTEN", "FORSCHUNG", "WISSENSCHAFTEN", "EV",
    # Sector & Business Descriptors
    "QUANTUM", "QUANTIQUE", "COMPUTING", "COMPUTER", "CALCULATING", "INFORMATION", "SYSTEMS", "SYSTEM",
    "NETWORK", "NETWORKS", "ELECTRONIC", "ELECTRONICS", "ELECTRIC", "POWER", "GRID", "ENERGY",
    "DIGITAL", "INTELLIGENT", "SMART", "DATA", "SOFTWARE", "SOLUTIONS", "SOLUTION",
    "INFRASTRUCTURE", "INSTRUMENT", "INSTRUMENTS", "DEVICES", "LICENSING", "PERMISSION",
    "INNOVATION", "INNOVATIONS", "VENTURES", "CAPITAL", "INVESTMENT", "PORTFOLIO",
    "INTELLECTUAL", "PROPERTY", "FINANCIAL", "INDUSTRY", "INDUSTRIAL", "DEVELOPMENT",
    "TECHNOLOGY", "MANUFACTURING", "APPLICATION", "APPLIED", "SERVICES", "SERVICE",
    "MACHINES", "EQUIPMENT", "AEROSPACE", "SPACE", "VISION", "BIOLOGICAL", "HEALTH",
    "MEDICINE", "MEDICAL", "ATOMIC", "ALTERNATIVE", "MOBILE", "MOBILITY", "TELECOM",
    "TELECOMMUNICATIONS", "COMMUNICATION", "COMMUNICATIONS", "ENGINEERING", "AGRICULTURAL",
    "MARITIME", "PETROLEUM", "MINERALS", "JIAOTONG", "EDUCATION", "HIGHER", "OPTICS",
    "OPTO", "PHOTONICS", "NANO", "NANOSCIENCE", "NANOTECHNOLOGY", "TOOLS", "CLOUD",
    "STORAGE", "SECURITY", "DEFENCE", "DEFENSE", "MILITARY", "PLA", "ARMY", "NAVY", "FORCE",
    "CIRCUITS", "INTEGRATED", "TIME", "PLATFORMS", "PLATFORM", "MOTOR", "INTELLIGENCE", "ARTIFICIAL",
    # Countries, Regions & Major Cities
    "CHINA", "CHINESE", "JAPAN", "KOREA", "EUROPE", "EUROPEAN", "AMERICA", "AMERICAN",
    "UNITED", "STATES", "KINGDOM", "UK", "US", "DEUTSCHLAND", "GERMANY", "FRANCE", "AUSTRALIA",
    "BEIJING", "SHANGHAI", "SHENZHEN", "GUANGDONG", "ANHUI", "JIANGSU", "ZHEJIANG",
    "SICHUAN", "HUBEI", "HUNAN", "HENAN", "HEBEI", "FUJIAN", "SHANDONG", "SHANXI",
    "SHAANXI", "LIAONING", "JILIN", "HEILONGJIANG", "YUNNAN", "GUIZHOU", "GANSU",
    "QINGHAI", "HAINAN", "NINGXIA", "XINJIANG", "XIZANG", "GUANGXI", "INNER", "MONGOLIA",
    "HEFEI", "NANJING", "SUZHOU", "HANGZHOU", "WUHAN", "CHENGDU", "CHONGQING", "DALIAN",
    "HARBIN", "TIANJIN", "XIAN", "CHANGCHUN", "SHENYANG", "ZHENGZHOU", "CHANGSHA",
    "FUZHOU", "XIAMEN", "JINAN", "KUNMING", "TAIYUAN", "DONGGUAN", "QINGDAO", "WUXI",
    "NORTH", "SOUTH", "EAST", "WEST", "NORTHEAST", "NORTHWEST", "SOUTHEAST", "SOUTHWEST",
    "NEW", "COLORADO", "FLORIDA", "ARIZONA", "MICHIGAN", "OHIO", "PENN", "TEXAS", "CALIFORNIA",
    "YORK", "WASHINGTON", "VIRGINIA"
}


def _core_tokens(s: str) -> set:
    toks = str(s).split()
    core = {t for t in toks if t not in BOILERPLATE_CORE and len(t) >= 2}
    return core if core else set(toks)


def _idf_jaccard_core(a: str, b: str) -> float:
    sa, sb = _core_tokens(a), _core_tokens(b)
    if not sa or not sb:
        return 0.0
    inter = sa & sb
    union = sa | sb
    w_inter = sum((idf_map.get(t, 1.0)) ** 2 for t in inter)
    w_union = sum((idf_map.get(t, 1.0)) ** 2 for t in union)
    return float(w_inter / w_union) if w_union > 0 else 0.0


def rectify_stage1_cluster(members_df: pd.DataFrame) -> list:
    """Split a Stage 1 cluster by entity_type and tightened core-lexical + semantic graph."""
    sub_clusters = []
    for etype, grp in members_df.groupby("entity_type"):
        u_names = sorted(grp["clean"].unique())
        if len(u_names) < MIN_CLUSTER_SIZE_TO_SPLIT or etype == "uncertain":
            for u in u_names:
                sub_clusters.append((etype, [u]))
            continue

        G_sub = nx.Graph()
        G_sub.add_nodes_from(u_names)
        for i in range(len(u_names)):
            for j in range(i + 1, len(u_names)):
                a, b = u_names[i], u_names[j]
                cos = float(np.dot(emb_map[a], emb_map[b]))
                jac_core = _idf_jaccard_core(a, b)
                ed_sort = rf_token_sort(a, b) / 100.0
                if jac_core < CORE_JAC_MIN and ed_sort < CORE_EDIT_MIN:
                    continue
                score = 0.34 * cos + 0.33 * jac_core + 0.33 * ed_sort
                if score >= th_otsu:
                    G_sub.add_edge(a, b, weight=score)

        comms = nx_comm.louvain_communities(G_sub, weight="weight", resolution=LOUVAIN_RESOLUTION_S2, seed=LOUVAIN_SEED)
        for c in comms:
            sub_clusters.append((etype, sorted(c)))
    return sub_clusters


# Execute Stage 2 partitioning across all Stage 1 clusters
stage2_records = []
for hid, grp in df_work.groupby("harm_id"):
    for etype, members in rectify_stage1_cluster(grp):
        stage2_records.append((etype, members))

stage2_records.sort(key=lambda item: (-sum(name_weights.get(m, 1) for m in item[1]), min(item[1])))

clean_to_id_s2 = {}
clean_to_canon_s2 = {}
for idx, (etype, members) in enumerate(stage2_records, start=1):
    new_id = f"HARM_{idx:05d}"
    best_canon = min(members, key=lambda s: (-name_weights.get(s, 1), len(s), s))
    for m in members:
        clean_to_id_s2[m] = new_id
        clean_to_canon_s2[m] = best_canon

df_work["harm_id_final"] = df_work["clean"].map(clean_to_id_s2)
df_work["harmonized_name_final"] = df_work["clean"].map(clean_to_canon_s2)

print(f"Stage 2 Rectification complete: {df_work['harm_id'].nunique():,} -> {df_work['harm_id_final'].nunique():,} clusters.")

Stage 2 Rectification complete: 1,372 -> 2,057 clusters.


## 5. Stage 3: Core-Anchor & Subsidiary Consolidation (Recall Recovery)

Strict Stage 2 lexical guards prevent false cross-company merges, but can leave regional subsidiaries, word-order permutations, and acronym expansions separated (e.g., `IBM` vs. `IBM DEUTSCHLAND` / `INTERNATIONAL BUSINESS MACHINES`, `TOSHIBA` vs. `TOSHIBA RESEARCH EUROPE`, `STANFORD UNIV` vs. `LELAND STANFORD JUNIOR UNIV`).

This final consolidation step merges Stage 2 clusters **within compatible entity categories** whenever their canonical names satisfy one of three high-precision symbolic rules:
1. **Rare Brand-Anchor Subset:** After removing institutional, sector, and geographic stopwords (`ANCHOR_STOPWORDS`, retaining US state names as distinct identifiers for universities), one cluster's distinguishing tokens are a strict subset of another's and share at least one rare proper identifier ($\text{df} \le 12$), while preventing universities from merging across distinct institutional categories (`UNIV` vs. `POLYTECHNIC` / `SCHOOL MINES`).
2. **Core Token-Sorted Fuzzy Match:** Distinguishing cores ($\ge 8$ characters) match with $\ge 90\%$ `token_sort_ratio` (`PSIQUANTUM` $\leftrightarrow$ `PSYQUANTUM`).
3. **Exact Acronym Expansion:** A 3–6 letter uppercase acronym matches the exact word initials of another entity in the same sector (`IBM` $\leftrightarrow$ `INTERNATIONAL BUSINESS MACHINES`, `NTT` $\leftrightarrow$ `NIPPON TELEGRAPH TELEPHONE`).

Canonical display names (`harmonized_name_final`) are derived directly from the final merged cluster members weighted by PATSTAT filing frequency (`name_weights`).

In [7]:
US_STATES = {"COLORADO", "FLORIDA", "ARIZONA", "MICHIGAN", "OHIO", "PENN", "TEXAS", "CALIFORNIA", "WASHINGTON", "VIRGINIA"}

ANCHOR_STOPWORDS = {
    # Institutional & Legal
    "UNIV", "INST", "INSTITUTE", "COLLEGE", "ACADEMY", "SCHOOL", "HOCHSCHULE", "ECOLE",
    "POLITECNICA", "POLYTECHNICAL", "POLYTECHNIC", "UNIVERSITETT", "UNIVERSITAET", "UNIVERSITAT",
    "UNIVERSITE", "UNIVERSIDAD", "UNIVERSITA", "UNIVERSITET", "TECHNISCHE", "VOCATIONAL",
    "TECH", "SCIENCE", "RESEARCH", "LAB", "CENTER", "GROUP", "CORP", "COMPANY", "COLTD",
    "HOLDINGS", "INTERNATIONAL", "NATIONAL", "STATE", "PUBLIC", "ROYAL", "NORMAL", "GENERAL",
    "CENTRAL", "FEDERAL", "LIMITED", "RESPONSIBILITY", "KORPERSCHAFT", "OFFENTLICHEN", "RECHTS",
    "BOARD", "REGENTS", "TRUSTEES", "FOUNDATION", "ASSOCIATION", "PRESIDENT", "FELLOWS",
    "HOSPITAL", "DEPARTMENT", "MINISTRY", "GOVERNMENT", "AGENCY", "SECRETARY", "DIRECTOR",
    "COMMITTEE", "COUNCIL", "INSTITUTION", "ADMINISTRATIVE", "INDEPENDENT",
    "FEDERALNOE", "GOSUDARSTVENNOE", "BYUDZHETNOE", "OBRAZOVATELNOE", "UCHREZHDENIE", "VYSSHEGO", "OBRAZOVANIYA",
    "GESELLSCHAFT", "FORDERUNG", "ANGEWANDTEN", "FORSCHUNG", "WISSENSCHAFTEN", "EV",
    "DEGLI", "STUDI", "NORMALE", "SUPERIEURE", "POSTS", "COOPERATION", "ACADEMIC", "ACADEMIA",
    # Sector & Business Descriptors
    "QUANTUM", "QUANTIQUE", "COMPUTING", "COMPUTER", "CALCULATING", "INFORMATION", "SYSTEMS", "SYSTEM",
    "NETWORK", "NETWORKS", "ELECTRONIC", "ELECTRONICS", "ELECTRIC", "POWER", "GRID", "ENERGY",
    "DIGITAL", "INTELLIGENT", "SMART", "DATA", "SOFTWARE", "SOLUTIONS", "SOLUTION",
    "INFRASTRUCTURE", "INSTRUMENT", "INSTRUMENTS", "DEVICES", "LICENSING", "PERMISSION",
    "INNOVATION", "INNOVATIONS", "VENTURES", "CAPITAL", "INVESTMENT", "PORTFOLIO",
    "INTELLECTUAL", "PROPERTY", "FINANCIAL", "INDUSTRY", "INDUSTRIAL", "DEVELOPMENT",
    "TECHNOLOGY", "MANUFACTURING", "APPLICATION", "APPLIED", "SERVICES", "SERVICE",
    "MACHINES", "EQUIPMENT", "AEROSPACE", "SPACE", "VISION", "BIOLOGICAL", "HEALTH",
    "MEDICINE", "MEDICAL", "ATOMIC", "ALTERNATIVE", "MOBILE", "MOBILITY", "TELECOM",
    "TELECOMMUNICATIONS", "COMMUNICATION", "COMMUNICATIONS", "ENGINEERING", "AGRICULTURAL",
    "MARITIME", "PETROLEUM", "MINERALS", "JIAOTONG", "EDUCATION", "HIGHER", "OPTICS",
    "OPTO", "PHOTONICS", "NANO", "NANOSCIENCE", "NANOTECHNOLOGY", "TOOLS", "CLOUD",
    "STORAGE", "SECURITY", "DEFENCE", "DEFENSE", "MILITARY", "PLA", "ARMY", "NAVY", "FORCE",
    "MOTOR", "PLATFORMS", "PLATFORM", "INTELLIGENCE", "ARTIFICIAL", "CIRCUITS", "INTEGRATED", "TIME",
    "SILICON", "WAVE", "GLOBAL", "STAR", "GREAT", "SUPER", "ADVANCED", "FUTURE", "OPEN", "SECOND", "MATRIX",
    # Countries, Regions & Major Cities
    "CHINA", "CHINESE", "JAPAN", "KOREA", "EUROPE", "EUROPEAN", "AMERICA", "AMERICAN", "UNITED", "STATES",
    "KINGDOM", "UK", "US", "DEUTSCHLAND", "GERMANY", "FRANCE", "AUSTRALIA", "CANADA", "ISRAEL", "SWITZERLAND", "NETHERLANDS",
    "BEIJING", "SHANGHAI", "SHENZHEN", "GUANGDONG", "ANHUI", "JIANGSU", "ZHEJIANG", "SICHUAN", "HUBEI",
    "HUNAN", "HENAN", "HEBEI", "FUJIAN", "SHANDONG", "SHANXI", "SHAANXI", "LIAONING", "JILIN", "HEILONGJIANG",
    "YUNNAN", "GUIZHOU", "GANSU", "QINGHAI", "HAINAN", "NINGXIA", "XINJIANG", "XIZANG", "GUANGXI", "INNER", "MONGOLIA",
    "HEFEI", "NANJING", "SUZHOU", "HANGZHOU", "WUHAN", "CHENGDU", "CHONGQING", "DALIAN", "HARBIN",
    "TIANJIN", "XIAN", "CHANGCHUN", "SHENYANG", "ZHENGZHOU", "CHANGSHA", "FUZHOU", "XIAMEN", "JINAN",
    "KUNMING", "TAIYUAN", "DONGGUAN", "QINGDAO", "WUXI",
    "NORTH", "SOUTH", "EAST", "WEST", "NORTHEAST", "NORTHWEST", "SOUTHEAST", "SOUTHWEST", "NEW", "YORK",
    "OXFORD", "CAMBRIDGE", "TOKYO", "SEOUL", "PARIS", "BERLIN", "MUNICH", "MUNCHEN", "ZURICH", "VIENNA", "WIEN", "DELFT"
} | US_STATES

INST_TYPE_TOKENS = {"UNIV", "POLYTECHNIC", "COLLEGE", "SCHOOL", "MINES", "INST", "ACADEMY", "HOSPITAL"}


def _extract_anchor(name: str, etype: str) -> set:
    stops = (ANCHOR_STOPWORDS - US_STATES) if etype == "university" else ANCHOR_STOPWORDS
    return {t for t in str(name).split() if t not in stops and len(t) >= 3}


def _is_exact_acronym(acro: str, full_str: str) -> bool:
    if not (3 <= len(acro) <= 6 and acro.isalpha() and len(acro.split()) == 1):
        return False
    words = [w for w in full_str.split() if w != acro]
    return len(words) == len(acro) and acro == "".join(w[0] for w in words)


stage2_clusters = (
    df_work.groupby(["harm_id_final", "entity_type", "harmonized_name_final"])
    .agg(weight=("clean", lambda s: sum(name_weights.get(x, 1) for x in s)))
    .reset_index()
)

RARE_ANCHOR_IDF = np.log(N_CORPUS / RARE_ANCHOR_MAX_DF)
G_anchor = nx.Graph()
G_anchor.add_nodes_from(stage2_clusters["harm_id_final"])

for etype, grp in stage2_clusters.groupby("entity_type"):
    if etype == "uncertain":
        continue
    rows = list(grp.itertuples())
    for i in range(len(rows)):
        a = rows[i].harmonized_name_final
        ca = _extract_anchor(a, etype)
        ca_str = " ".join(sorted(ca))
        ia = set(a.split()) & INST_TYPE_TOKENS
        for j in range(i + 1, len(rows)):
            b = rows[j].harmonized_name_final
            cb = _extract_anchor(b, etype)
            cb_str = " ".join(sorted(cb))
            ib = set(b.split()) & INST_TYPE_TOKENS

            if etype == "university" and ia and ib and ia != ib:
                continue

            do_merge = False
            if ca and cb and (ca <= cb or cb <= ca):
                shared = ca & cb
                if any(idf_map.get(t, 0.0) >= RARE_ANCHOR_IDF for t in shared):
                    do_merge = True

            if not do_merge and len(ca_str) >= 8 and len(cb_str) >= 8:
                if rf_token_sort(ca_str, cb_str) >= CORE_FUZZY_MIN:
                    do_merge = True

            if not do_merge and (_is_exact_acronym(a, b) or _is_exact_acronym(b, a)):
                do_merge = True

            if do_merge:
                G_anchor.add_edge(rows[i].harm_id_final, rows[j].harm_id_final)

merged_components = list(nx.connected_components(G_anchor))
old_id_to_weight = stage2_clusters.set_index("harm_id_final")["weight"].to_dict()
old_id_to_canon = stage2_clusters.set_index("harm_id_final")["harmonized_name_final"].to_dict()

merged_components.sort(
    key=lambda comp: (-sum(old_id_to_weight[hid] for hid in comp), min(old_id_to_canon[hid] for hid in comp))
)

old_to_new_id = {}
for idx, comp in enumerate(merged_components, start=1):
    new_hid = f"HARM_{idx:05d}"
    for hid in comp:
        old_to_new_id[hid] = new_hid

df_work["harm_id_final"] = df_work["harm_id_final"].map(old_to_new_id)


def _pick_canonical(sub_clean: pd.Series) -> str:
    unique_members = sub_clean.unique()
    return min(unique_members, key=lambda s: (-name_weights.get(s, 1), len(s), s))


canon_map = df_work.groupby("harm_id_final")["clean"].apply(_pick_canonical)
df_work["harmonized_name_final"] = df_work["harm_id_final"].map(canon_map)

print(f"Stage 3 Anchor Consolidation complete: {len(stage2_clusters):,} -> {df_work['harm_id_final'].nunique():,} final entities.")

Stage 3 Anchor Consolidation complete: 2,057 -> 1,880 final entities.


## 6. Production Export & Quality Diagnostics

Exports two deliverables:
1. **`harmonization_lookup_rectified.csv`:** Deduplicated lookup table mapping every organizational `psn_name` to `clean`, `entity_type`, `harm_id_final`, and `harmonized_name_final`.
2. **`qc_applicants_harmonized_PROD.csv`:** The complete input PATSTAT table (`raw_df`) enriched with `psn_sector_imputed`, `clean`, `entity_type`, `harm_id_final`, and `harmonized_name_final` via a left join (retaining individual inventor rows with `NaN` entity IDs while harmonizing 100% of organizational records).

Prints entity-type consolidation rates and inspects the largest resulting clusters for final QA verification.

In [8]:
lookup_out = (
    df_work[[NAME_COL, "clean", "entity_type", "harm_id_final", "harmonized_name_final"]]
    .drop_duplicates(subset=[NAME_COL])
    .sort_values(["harm_id_final", NAME_COL])
)
lookup_out.to_csv(OUTPUT_FINAL_LOOKUP, index=False)

# Left-join back onto full PATSTAT table (preserves all rows; individuals keep NaN harm_id_final)
full_harmonized = raw_df.merge(lookup_out, on=NAME_COL, how="left")
full_harmonized.to_csv(OUTPUT_FINAL_DATA, index=False)

n_org_rows = full_harmonized["harm_id_final"].notna().sum()
n_ind_rows = full_harmonized["harm_id_final"].isna().sum()

print(f"Saved lookup table -> {OUTPUT_FINAL_LOOKUP} ({len(lookup_out):,} unique org names -> {lookup_out['harm_id_final'].nunique():,} entities)")
print(f"Saved full dataset -> {OUTPUT_FINAL_DATA} ({len(full_harmonized):,} total rows: {n_org_rows:,} org rows harmonized, {n_ind_rows:,} individual/empty rows)")

print("\n--- Consolidation Summary by Entity Type ---")
summary = lookup_out.groupby("entity_type").agg(
    n_raw=(NAME_COL, "nunique"),
    n_clean=("clean", "nunique"),
    n_entities=("harm_id_final", "nunique"),
)
summary["consolidation_%"] = (100 * (1 - summary["n_entities"] / summary["n_clean"])).round(1)
print(summary.to_string())

print("\n--- Top 15 Largest Harmonized Entities (by patent rows) ---")
print(full_harmonized["harmonized_name_final"].value_counts().head(15).to_string())

Saved lookup table -> outputs/harmonization_lookup_rectified.csv (3,279 unique org names -> 1,880 entities)
Saved full dataset -> outputs/qc_applicants_harmonized_PROD.csv (20,895 total rows: 17,751 org rows harmonized, 3,144 individual/empty rows)

--- Consolidation Summary by Entity Type ---
                  n_raw  n_clean  n_entities  consolidation_%
entity_type                                                  
company            2186     1661        1246             25.0
gov / non-profit    287      250         160             36.0
uncertain             1        1           1              0.0
university          805      738         473             35.9

--- Top 15 Largest Harmonized Entities (by patent rows) ---
harmonized_name_final
IBM INTERNATIONAL BUSINESS MACHINES    1216
ORIGIN QUANTUM COMPUTING TECH HEFEI    1059
BEIJING BAIDU NETCOM SCIENCE TECH       415
MICROSOFT TECH LICENSING                288
GOOGLE                                  274
QUANTINUUM                    